# SAR surface-water mapping

This notebook runs the locked four-family classical experiment. It audits supplied VV/VH pairs, freezes source splits, calibrates on validation only, then exports held-out water metrics and a separate synthetic PSNR benchmark. The organizer subset is still an external input. No result or fresh Colab runtime is claimed by this unexecuted notebook.

The `full` profile here is the complete **classical** experiment, including installation, data access, audit, validation, scoring, figures and export. Neural training is disabled. Remotely trained checkpoints belong to a separately reported `checkpoint-evaluation` profile; its inference time cannot establish full training reproduction within 30 minutes. Observed surface water does not distinguish new flooding from permanent water.

In [ ]:
from time import perf_counter
NOTEBOOK_START = perf_counter()  # Run first: includes setup and documented access below.
from pathlib import Path
from datetime import datetime, timezone
import csv, json, os, platform, subprocess, sys, shutil
STAGES = {}
STARTED_UTC = datetime.now(timezone.utc).isoformat()
if sys.version_info < (3, 12):
    raise RuntimeError('Python >=3.12 for the pinned Colab dependencies is required. Select a compatible Colab runtime.')


Set these inputs before Run All. Supply a long-form inventory (`data/README.md`), a CSV of homogeneous **training** ROIs for effective looks, and independently preselected held-out ENL ROIs. Coordinates use half-open `y0:y1,x0:x1`; save `patch_id,y0,y1,x0,x1,suitability,roi_class`. Record ROI suitability from raw images before inspecting filter outputs. Missing held-out ROIs remain an explicit missing ENL result.

For upstream development only, `DOWNLOAD_UPSTREAM_DEVELOPMENT=True` fetches exactly the separately identified 100 hand-labeled pairs. It does not become the organizer subset. Download time is included. Otherwise mount/upload your authorized data before supplying its inventory path and disclose that access mechanism. No credentials belong in this notebook.

In [ ]:
REPOSITORY = 'https://github.com/kaustubh-dot/sar-water-mapping.git'
REPO_REF = 'implementation/remaining-phases'  # Replace with reviewed commit SHA for an exact reproduction.
REPO = Path('/content/sar-water-mapping') if 'google.colab' in sys.modules else Path.cwd()
DOWNLOAD_UPSTREAM_DEVELOPMENT = False
INVENTORY = None  # Set Path('/content/.../inventory.csv') for your paired data.
SUPPLIED_SPLITS = None  # Optional Path to supplied official/organizer patch_id,split CSV.
TRAINING_ROIS = None  # Required: Path to raw-training homogeneous ROI CSV.
HELDOUT_ENL_ROIS = None  # Optional: Path to preselected held-out homogeneous ROI CSV.
DATA_ACCESS_DESCRIPTION = 'user-supplied local or mounted files; disclose upload/mount timing separately'
PROFILE = 'full'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')


In [ ]:
def checkout_source(repository, reference, destination):
    """Resolve a branch, tag or full commit SHA and verify this clean checkout."""
    destination = Path(destination)
    if not reference or reference != reference.strip():
        raise ValueError('Supply an explicit branch, tag or full commit SHA.')
    existing_checkout = destination.exists()
    if not existing_checkout:
        subprocess.run(['git', 'clone', '--no-checkout', repository, str(destination)], check=True)
    elif not (destination / '.git').exists():
        raise RuntimeError('Existing source directory is not a Git checkout.')
    prefix = ['git', '-C', str(destination)]
    actual_remote = subprocess.check_output(prefix + ['remote', 'get-url', 'origin'], text=True).strip()
    if actual_remote != repository:
        raise RuntimeError('Existing checkout origin does not match the requested repository.')
    if existing_checkout and subprocess.check_output(prefix + ['status', '--porcelain'], text=True).strip():
        raise RuntimeError('Existing checkout has uncommitted changes; preserve them and use a clean destination.')
    # Fetching the requested reference also supports a reachable full commit SHA.
    subprocess.run(prefix + ['fetch', '--no-tags', 'origin', reference], check=True)
    expected = subprocess.check_output(prefix + ['rev-parse', '--verify', 'FETCH_HEAD^{commit}'], text=True).strip()
    if len(reference) == 40 and all(c in '0123456789abcdefABCDEF' for c in reference):
        if expected.lower() != reference.lower():
            raise RuntimeError('Fetched commit does not match the requested full SHA.')
    subprocess.run(prefix + ['checkout', '--detach', expected], check=True)
    actual = subprocess.check_output(prefix + ['rev-parse', 'HEAD'], text=True).strip()
    if actual != expected:
        raise RuntimeError('Executed checkout does not match the resolved requested reference.')
    return actual

setup_start = perf_counter()
SOURCE_SHA = checkout_source(REPOSITORY, REPO_REF, REPO)
os.chdir(REPO)
print('Requested source reference:', REPO_REF)
print('Verified executed source commit:', SOURCE_SHA)
lock = REPO / 'requirements-colab.txt'
if not lock.is_file():
    raise FileNotFoundError('The tested core dependency lock requirements-colab.txt is missing from this checkout.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(lock)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], check=True)
# Keep Colab's preinstalled torch/CUDA untouched; this classical profile does not use them.
STAGES['installation_seconds'] = perf_counter() - setup_start


In [ ]:
access_start = perf_counter()
if DOWNLOAD_UPSTREAM_DEVELOPMENT:
    subprocess.run([sys.executable, 'scripts/prepare_development_data.py', '--data-root', 'data/development-v1'], check=True)
    INVENTORY = REPO / 'data/development-v1/inventory.csv'
    DATA_ACCESS_DESCRIPTION = 'bounded upstream 100 hand-labeled development pairs; not organizer subset'
if INVENTORY is None or not Path(INVENTORY).is_file():
    raise FileNotFoundError('Set INVENTORY to your actual long-form CSV, or explicitly enable the upstream development download. No data are fabricated.')
if TRAINING_ROIS is None or not Path(TRAINING_ROIS).is_file():
    raise FileNotFoundError('Set TRAINING_ROIS to recorded homogeneous raw-training rectangles before calibration; no automatic L=1 fallback is allowed.')
INVENTORY, TRAINING_ROIS = Path(INVENTORY).resolve(), Path(TRAINING_ROIS).resolve()
if HELDOUT_ENL_ROIS is not None:
    HELDOUT_ENL_ROIS = Path(HELDOUT_ENL_ROIS).resolve()
    if not HELDOUT_ENL_ROIS.is_file():
        raise FileNotFoundError(HELDOUT_ENL_ROIS)
STAGES['documented_data_access_seconds'] = perf_counter() - access_start
RUN_ROOT = REPO / 'outputs' / ('colab-' + RUN_ID)
AUDIT = RUN_ROOT / 'audit'
CALIBRATION = RUN_ROOT / 'calibration'
FROZEN_CONFIG = RUN_ROOT / 'experiment.json'
RESULTS = RUN_ROOT / 'results'


In [ ]:
checks_start = perf_counter()
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_data.py', 'tests/test_metrics.py', 'tests/test_filters.py', 'tests/test_segment.py', 'tests/test_benchmark.py', 'tests/test_evaluate.py', '--basetemp', str(RUN_ROOT / 'pytest')], check=True)
STAGES['scientific_checks_seconds'] = perf_counter() - checks_start


In [ ]:
audit_start = perf_counter()
command = [sys.executable, '-m', 'sar_water.data', str(INVENTORY), '--output', str(AUDIT), '--contact-sheet']
if SUPPLIED_SPLITS is not None:
    command.extend(['--splits', str(Path(SUPPLIED_SPLITS).resolve())])
subprocess.run(command, check=True)
MANIFEST, SPLITS = AUDIT / 'manifest.csv', AUDIT / 'splits.csv'
STAGES['audit_seconds'] = perf_counter() - audit_start
from IPython.display import display, Image
display(Image(filename=str(AUDIT / 'contact_sheet.png')))


Calibration uses training ROIs for effective looks and validation masks for the common channel/threshold rule, bounded filter parameters and selected filtered input. It creates a new versioned config bound to the audited manifest and split hashes. Final evaluation refuses unfrozen or mismatched inputs. Otsu may split unimodal dry land into false water; constant images use the recorded calibrated fallback. The full per-band validation table and cleanup ablation are retained.

In [ ]:
calibration_start = perf_counter()
calibrator = REPO / 'scripts/calibrate_classical.py'
if not calibrator.is_file():
    raise FileNotFoundError('Validation calibration implementation is missing from this checkout; use the reviewed implementation branch.')
subprocess.run([sys.executable, str(calibrator), '--manifest', str(MANIFEST), '--splits', str(SPLITS), '--rois', str(TRAINING_ROIS), '--config', str(FROZEN_CONFIG), '--output', str(CALIBRATION)], check=True)
STAGES['training_roi_and_validation_calibration_seconds'] = perf_counter() - calibration_start
frozen = json.loads(FROZEN_CONFIG.read_text())
print(json.dumps(frozen['real_data'], indent=2))


In [ ]:
from sar_water.evaluate import run_evaluation
if PROFILE != 'full':
    raise ValueError('This notebook executes the full classical profile. Add separately verified neural checkpoint provenance before claiming checkpoint evaluation.')
evaluation_start = perf_counter()
runtime = run_evaluation(MANIFEST, SPLITS, FROZEN_CONFIG, RESULTS,
    enl_rois=HELDOUT_ENL_ROIS, profile=PROFILE,
    runtime_type='colab_cpu' if 'google.colab' in sys.modules else 'local_cpu_not_colab',
    external_setup_seconds=evaluation_start-NOTEBOOK_START, synthetic=True, figures=True)
STAGES['frozen_real_synthetic_evaluation_export_seconds'] = perf_counter() - evaluation_start


In [ ]:
def show_csv(path):
    from IPython.display import HTML, display
    import html
    with Path(path).open(newline='', encoding='utf-8') as stream:
        rows = list(csv.reader(stream))
    display(HTML('<div style="overflow:auto"><table>' + ''.join('<tr>' + ''.join('<td style="padding:4px;border:1px solid #ddd">' + html.escape(value) + '</td>' for value in row) + '</tr>' for row in rows) + '</table></div>'))
print('Real held-out surface-water segmentation; primary cleanup disabled')
show_csv(RESULTS / 'results_real.csv')
print('Synthetic held-out PSNR; simulated known references, never real SAR ground truth')
show_csv(RESULTS / 'results_synthetic.csv')
print('ENL:', runtime['enl_status'])
show_csv(RESULTS / 'enl_summary.csv')
figure_paths = sorted((RESULTS / 'figures').glob('*.png'))
for figure_path in figure_paths[:8]:
    display(Image(filename=str(figure_path)))


Inspect the aligned VV/VH images, errors and raw/filtered power ratio for lost narrow water and boundaries; report failures even if pooled scores improve. Per-patch predictions never alter the scoring mask. ENL reports undefined ROI counts and uses fixed positive-power regions; a smoother image is not necessarily a more faithful one.

The exports retain the unchanged raw baseline, primary four-row table, separate cleanup ablation, pooled counts, fixed water-positive macro set, dry-patch false positives, per-band radiometric changes, synthetic selection/test rows and provenance. A shorter Colab neural training schedule would be a separate experiment; no neural accuracy or remote-training time is implied here.

In [ ]:
from sar_water.data import sha256_file

def export_bundle(run_root, evidence, notebook_start, stages):
    """Archive once; save authoritative post-archive timing in a sibling sidecar."""
    export_start = perf_counter()
    run_root = Path(run_root)
    archive_base = str(run_root.parent / run_root.name)
    timing_path = run_root.parent / (run_root.name + '.runtime.json')
    manifest = dict(evidence, stages_seconds=dict(stages),
        authoritative_timing_sidecar=timing_path.name,
        timing_note='The sibling sidecar is written after the final archive and hash; it is intentionally outside this archive.')
    (run_root / 'bundle_manifest.json').write_text(json.dumps(manifest, indent=2, allow_nan=False) + '\n', encoding='utf-8')
    archive = shutil.make_archive(archive_base, 'zip', run_root)
    archive_hash = sha256_file(archive)
    stages['bundle_export_seconds'] = perf_counter() - export_start
    final = dict(evidence, stages_seconds=dict(stages), archive_path=str(archive),
        archive_sha256=archive_hash, authoritative_timing_sidecar=timing_path.name,
        total_wall_seconds=perf_counter() - notebook_start,
        timing_scope='First executable cell through the only archive write and archive checksum; excludes only timing-sidecar serialization and subsequent display.')
    final['measured_below_30_minutes'] = final['total_wall_seconds'] < 1800
    timing_path.write_text(json.dumps(final, indent=2, allow_nan=False) + '\n', encoding='utf-8')
    return archive, timing_path, final

notebook_evidence = dict(started_utc=STARTED_UTC, profile=PROFILE, neural_training_enabled=False,
    runtime_type=runtime['runtime_type'], source_commit_sha=SOURCE_SHA, requested_source_ref=REPO_REF,
    dependency_lock_sha256=sha256_file(REPO / 'requirements-colab.txt'),
    inventory_sha256=sha256_file(INVENTORY), training_rois_sha256=sha256_file(TRAINING_ROIS),
    data_access=DATA_ACCESS_DESCRIPTION, external_upload_or_mount_seconds=None,
    fresh_runtime_attestation='unverified; record actual fresh runtime and access conditions before submission')
archive, timing_path, notebook_evidence = export_bundle(RUN_ROOT, notebook_evidence, NOTEBOOK_START, STAGES)
print(json.dumps(notebook_evidence, indent=2))
print('Bundle:', archive)
print('Authoritative final timing sidecar (outside the ZIP):', timing_path)
from IPython.display import FileLink
display(FileLink(archive))
display(FileLink(str(timing_path)))
